# UrbanFlow · January 2025 ingestion

Run all cells in a standard **CPU** Colab runtime. Approve Google Drive access when prompted. Keep `urbanflow-pipeline.zip` at the root of My Drive next to this notebook. No MongoDB credentials or paid runtime are needed.

The notebook orchestrates the reusable Python package; ingestion logic stays in `pipelines/`. Raw data, checksums and progress persist in `My Drive/UrbanFlow/data/raw`. Re-running verifies existing downloads.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import sys
import zipfile
from pathlib import Path

bundle = Path("/content/drive/MyDrive/urbanflow-pipeline.zip")
expected_sha256 = "62300f587c8a23b35f0cce2dd3a5ff5ccb20ed305c027807cf21a336cbc3ae65"
if not bundle.is_file():
    raise FileNotFoundError("Put urbanflow-pipeline.zip at the root of My Drive, then rerun.")
if hashlib.sha256(bundle.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Pipeline bundle checksum mismatch. Use the bundle supplied with this notebook."
    )
workspace = Path("/content/urbanflow").resolve()
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    for member in archive.infolist():
        if not (workspace / member.filename).resolve().is_relative_to(workspace):
            raise ValueError("Unsafe archive path")
    archive.extractall(workspace)
sys.path.insert(0, str(workspace / "pipelines" / "src"))
print("Verified reusable pipeline:", expected_sha256)

In [ ]:
import json

from urbanflow_etl.ingestion.download import download_dataset

config = json.loads((workspace / "configs" / "yellow-2025-01.json").read_text())
output = Path("/content/drive/MyDrive/UrbanFlow/data/raw")
manifest = download_dataset(config, output)
print(json.dumps(manifest, indent=2))

In [ ]:
progress = json.loads((output / "download-progress.json").read_text())
print(json.dumps(progress, indent=2))
print("Persistent data:", output)

## Local fallback

If the download stalls, run `node scripts/download-data.mjs` from the repository on Windows. Upload the files and `2025-01-manifest.json` from `data/raw/` to `My Drive/UrbanFlow/data/raw/`, then rerun this notebook. Existing files are checksum-verified before reuse.

This notebook prepares source data. Spark transformations and MongoDB publication are subsequent milestones; no analytical results have been fabricated.